In [0]:
import pyspark.sql.functions as f
from delta.tables import DeltaTable


#Paramêtros construção de tabelas
dbutils.widgets.text("tabela_origem", "zeta_project.bronze.tabela", "1. Tabela Bronze")
dbutils.widgets.text("tabela_destino", "zeta_project.silver.tabela", "2. Tabela Silver")


# Parametros para tratamento de colunas()
dbutils.widgets.text("col_inteiro", "", "Colunas Inteiras (separadas por vírgula)")
dbutils.widgets.text("col_decimal", "", "Colunas Decimais (separadas por vírgula)")
dbutils.widgets.text("col_timestamp", "", "Colunas Decimais (separadas por vírgula)")

tabela_origem = dbutils.widgets.get("tabela_origem")
tabela_destino = dbutils.widgets.get("tabela_destino")
#Leitura da tabela bronze
df = spark.read.table(tabela_origem)

# Cria lista com os tipos das colunas passada através de parametros

cols_int = [ci.strip() for ci in dbutils.widgets.get("col_inteiro").split(",") if ci.strip()]
cols_dec = [cd.strip() for cd in dbutils.widgets.get("col_decimal").split(",") if cd.strip()]
cols_times = [ct.strip() for ct in dbutils.widgets.get("col_timestamp").split(",") if ct.strip()]


# Tenta realizar as conversões solicitadas
for ci in cols_int:
    if ci in df.columns:
        ci_limpa = f.trim(f.col(ci))
        df = df.withColumn(ci_limpa, f.col(ci).cast("int"))
        
for cd in cols_dec:
    if cd in df.columns:
        cd_limpa = f.trim(f.col(cd))
        df = df.withColumn(cd_limpa, f.col(cd).cast("decimal(18,2)"))

for ct in cols_times:
    if ct in df.columns:
        df = df.withColumn(ct, f.col(ct))        
        ct_limpa = f.trim(f.col(ct))
# Inclusao de tentativa de cast devido a diferentes tipos de datas encontradas
        df = df.withColumn(ct, 
            f.coalesce(
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-M-d H:m:s")), 
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-MM-dd HH:mm:ss")),
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-MM-dd HH:mm")),
                f.try_to_timestamp(ct_limpa, f.lit("d/M/yyyy H:m:s")),
                f.try_to_timestamp(ct_limpa, f.lit("dd/MM/yyyy HH:mm")),
                f.try_to_timestamp(ct_limpa, f.lit("d/M/yyyy H:m")),
                f.try_to_timestamp(ct_limpa)
            )
        )
display(df)